In [ ]:
import subprocess
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name',
                                   '--format=csv,noheader']).decode().strip()
    print(f"GPU: {out}")
except Exception:
    print("GPUなし（CPUで実行します。時間はかかりますが完走します）")

import platform, sys
print(f"Python: {sys.version.split()[0]}")
print(f"OS: {platform.system()}")

In [ ]:
!pip install ultralytics -q
!pip install imageio imageio-ffmpeg -q
print("インストール完了")

In [ ]:
from ultralytics import YOLO
import urllib.request

model = YOLO('yolov8n.pt')

img_url = "https://ultralytics.com/images/bus.jpg"
img_path = "input.jpg"
urllib.request.urlretrieve(img_url, img_path)
print(f"画像ダウンロード完了: {img_path}")

results = model(img_path)
r = results[0]
boxes = r.boxes.xyxy.cpu().numpy()
classes = r.boxes.cls.cpu().numpy()
names = r.names
orig_h, orig_w = r.orig_shape

print(f"\n検出数: {len(boxes)}")
detections = []
for i, (box, cls) in enumerate(zip(boxes, classes)):
    label = names[int(cls)]
    x_center = (box[0] + box[2]) / 2 / orig_w
    y_center = (box[1] + box[3]) / 2 / orig_h
    detections.append({'label': label, 'box': box,
                       'x_center': x_center, 'y_center': y_center})
    print(f"  [{i}] {label}  座標=({x_center:.2f}, {y_center:.2f})")

unique_targets = {}
for d in detections:
    if d['label'] not in unique_targets:
        unique_targets[d['label']] = d

print(f"\nユニークなターゲット: {list(unique_targets.keys())}")

In [ ]:
import numpy as np

L1, L2 = 1.0, 1.0

def forward_kinematics(theta1, theta2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return x, y

def inverse_kinematics(x, y):
    r2 = x**2 + y**2
    cos_t2 = np.clip((r2 - L1**2 - L2**2) / (2*L1*L2), -1, 1)
    theta2 = -np.arccos(cos_t2)
    theta1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(theta2),
                                            L1 + L2*np.cos(theta2))
    return theta1, theta2

def map_to_arm(x_center, y_center):
    tx = np.clip((x_center - 0.5) * 1.6, -1.8, 1.8)
    ty = np.clip((0.5 - y_center) * 1.6, -1.8, 1.8)
    return tx, ty

task_labels = [lb for lb in ['person', 'bus', 'stop sign']
               if lb in unique_targets]
if not task_labels:
    task_labels = [list(unique_targets.keys())[0]]

all_labels = list(unique_targets.keys())

FPS = 20
FRAMES_PER_TASK = 360   # 18秒 × 4タスク = 72秒

tasks = []
for label in task_labels[:3]:
    d = unique_targets[label]
    tx, ty = map_to_arm(d['x_center'], d['y_center'])
    tasks.append({'title': f"Task {len(tasks)+1}: Detect '{label}'",
                  'label': label, 'target_x': tx, 'target_y': ty})

tasks.append({
    'title': f"Task {len(tasks)+1}: Track All Objects",
    'label': 'ALL',
    'all_targets': [
        {'label': lb,
         'x': map_to_arm(unique_targets[lb]['x_center'],
                         unique_targets[lb]['y_center'])[0],
         'y': map_to_arm(unique_targets[lb]['x_center'],
                         unique_targets[lb]['y_center'])[1]}
        for lb in all_labels],
})

for i, t in enumerate(tasks):
    print(f"Task {i+1}: {t['title']}")

TOTAL_FRAMES = FRAMES_PER_TASK * len(tasks)
print(f"\n合計フレーム数: {TOTAL_FRAMES}")
print(f"合計再生時間: {TOTAL_FRAMES/FPS:.1f}秒")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

theta1_init, theta2_init = np.pi/2, -np.pi/2

def smoothstep(t):
    return 0.5 - 0.5*np.cos(np.pi*t)

def get_state(frame_idx):
    task_idx = min(frame_idx // FRAMES_PER_TASK, len(tasks) - 1)
    local_frame = frame_idx % FRAMES_PER_TASK
    task = tasks[task_idx]
    t = local_frame / (FRAMES_PER_TASK - 1)

    if task['label'] != 'ALL':
        s = smoothstep(t)
        t1, t2 = inverse_kinematics(task['target_x'], task['target_y'])
        th1 = theta1_init + (t1 - theta1_init) * s
        th2 = theta2_init + (t2 - theta2_init) * s
        targets_show = [(task['target_x'], task['target_y'])]
    else:
        n = len(task['all_targets'])
        seg = 1.0 / n
        seg_idx = min(int(t / seg), n - 1)
        seg_t = (t - seg_idx*seg) / seg
        s = smoothstep(seg_t)
        tgt = task['all_targets'][seg_idx]
        t1, t2 = inverse_kinematics(tgt['x'], tgt['y'])
        th1 = theta1_init + (t1 - theta1_init) * s
        th2 = theta2_init + (t2 - theta2_init) * s
        targets_show = [(a['x'], a['y']) for a in task['all_targets']]

    return task, th1, th2, targets_show

fig, ax = plt.subplots(figsize=(7, 7))

def update(frame_idx):
    ax.clear()
    task, th1, th2, targets_show = get_state(frame_idx)

    x0, y0 = 0, 0
    x1, y1 = L1*np.cos(th1), L1*np.sin(th1)
    x2, y2 = forward_kinematics(th1, th2)

    ax.plot([x0, x1], [y0, y1], 'o-', lw=7, color='steelblue',
            markersize=13, zorder=3)
    ax.plot([x1, x2], [y1, y2], 'o-', lw=7, color='coral',
            markersize=13, zorder=3)

    for tx, ty in targets_show:
        ax.plot(tx, ty, '*', markersize=28, color='gold',
                markeredgecolor='orange', markeredgewidth=2, zorder=5)

    ax.plot([x2], [y2], 'o', markersize=15, color='red', zorder=4)

    ax.set_xlim(-2.3, 2.3)
    ax.set_ylim(-2.5, 2.3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f"AI × Robot Arm  |  Frame {frame_idx+1}/{TOTAL_FRAMES}",
                 fontsize=13)
    ax.text(0, 2.1, task['title'], ha='center', fontsize=15,
            weight='bold', color='navy',
            bbox=dict(boxstyle='round', facecolor='lightyellow',
                      edgecolor='orange'))

    progress = (frame_idx + 1) / TOTAL_FRAMES
    ax.barh(-2.35, 4.4, height=0.15, color='lightgray', zorder=2)
    ax.barh(-2.35, 4.4*progress, height=0.15, color='green', zorder=3)
    elapsed = (frame_idx + 1) / FPS
    ax.text(2.1, -2.35, f"{elapsed:.1f}s", ha='right', fontsize=10,
            color='darkgreen')

    if frame_idx == TOTAL_FRAMES - 1:
        ax.text(0, -2.2, "All Tasks Completed!",
                ha='center', fontsize=16, color='green', weight='bold')

anim = FuncAnimation(fig, update, frames=TOTAL_FRAMES, interval=50)
anim.save('robot_arm_full.gif', writer=PillowWriter(fps=FPS))
plt.close()
print("GIF保存完了: robot_arm_full.gif")
print(f"再生時間: {TOTAL_FRAMES/FPS:.1f}秒")

from IPython.display import Image as IPImage
IPImage('robot_arm_full.gif')

In [ ]:
import imageio.v2 as imageio
import os

reader = imageio.get_reader('robot_arm_full.gif')
frames = [frame for frame in reader]
reader.close()

imageio.mimsave('robot_arm_full.mp4', frames, fps=FPS)
size_mb = os.path.getsize('robot_arm_full.mp4') / 1024 / 1024
print(f"MP4保存完了: robot_arm_full.mp4 ({size_mb:.1f} MB)")

In [ ]:
from google.colab import files
files.download('robot_arm_full.mp4')
files.download('robot_arm_full.gif')